### Task

In [1]:
from pathlib import Path
import sys

import torchvision.transforms as transforms
from xy_dataset import XYDataset

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

TASK = 'road_following'

CATEGORIES = ['apex']

DATASETS = ['A', 'B']

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

datasets = {}
for name in DATASETS:
    datasets[name] = XYDataset(TASK + '_' + name, CATEGORIES, TRANSFORMS, random_hflip=True)

### Data Collection

### Model

In [3]:
import cv2
import ipywidgets
import traitlets
import threading
import time
from ipyevents import Event
from simulation import CarSimulator, bgr8_to_jpeg
from IPython.display import display

# initialize active dataset
dataset = datasets[DATASETS[0]]

# Reuse existing Car instance or close old environment before creating new one
if 'Car' in globals() and Car is not None:
    try:
        Car.close()
    except Exception:
        pass

Car = CarSimulator()

def stream_loop():
    global stream_active
    while stream_active:
        Car.run(steering_slider.value, throttle_slider.value)
        time.sleep(0.05)  # 20 FPS live simulation stream

def on_stream_change(change):
    global stream_active
    if change['new'] == 'Stream Live':
        if not stream_active:
            stream_active = True
            t = threading.Thread(target=stream_loop, daemon=True)
            t.start()
    else:
        stream_active = False
# create image preview
camera_widget = ipywidgets.Image(value=Car.value, format='jpeg', width=Car.obs.shape[1], height=Car.obs.shape[0])
snapshot_widget = ipywidgets.Image(width=Car.obs.shape[1], height=Car.obs.shape[0])
traitlets.dlink((Car, 'value'), (camera_widget, 'value'))

# create dataset widgets
dataset_widget = ipywidgets.Dropdown(options=DATASETS, description='dataset')
category_widget = ipywidgets.Dropdown(options=dataset.categories, description='category')
count_widget = ipywidgets.IntText(description='count')

# manually update counts at initialization
count_widget.value = dataset.get_count(category_widget.value)


# Pending click data storage
pending_data = {'image': None, 'x': None, 'y': None}

# Create Control & Action Buttons
save_button = ipywidgets.Button(description='Save Snapshot', button_style='success', icon='check', disabled=True)
step_button = ipywidgets.Button(description='Step Once', button_style='info', icon='play')
reset_button = ipywidgets.Button(description='Reset Track', button_style='warning', icon='refresh')
stream_toggle = ipywidgets.ToggleButtons(options=['Stream Off', 'Stream Live'], description='Stream Mode', value='Stream Off')

stream_toggle.observe(on_stream_change, names='value')

steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, value=0.0, step=0.02, description='Steering')
throttle_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, value=0.15, step=0.02, description='Throttle')

stream_active = False

# sets the active dataset
def set_dataset(change):
    global dataset
    dataset = datasets[change['new']]
    count_widget.value = dataset.get_count(category_widget.value)
dataset_widget.observe(set_dataset, names='value')

# update counts when we select a new category
def update_counts(change):
    count_widget.value = dataset.get_count(change['new'])
category_widget.observe(update_counts, names='value')




def on_step_clicked(b):
    Car.run(steering_slider.value, throttle_slider.value)

def on_reset_clicked(b):
    global stream_active
    stream_toggle.value = 'Stream Off'
    stream_active = False
    time.sleep(0.1)
    Car.reset()

def on_save_clicked(b):
    if pending_data['image'] is not None and pending_data['x'] is not None:
        # Save pending snapshot to disk
        dataset.save_entry(category_widget.value, pending_data['image'], pending_data['x'], pending_data['y'])
        count_widget.value = dataset.get_count(category_widget.value)
        # Disable save button after saving to avoid duplicate saves
        save_button.disabled = True

save_button.on_click(on_save_clicked)
step_button.on_click(on_step_clicked)
reset_button.on_click(on_reset_clicked)

controls_box = ipywidgets.VBox([
    ipywidgets.HBox([steering_slider, throttle_slider]),
    ipywidgets.HBox([save_button, step_button, reset_button, stream_toggle])
])

# Add click event listener to camera widget using ipyevents
im_event = Event(source=camera_widget, watched_events=['click'])

def handle_click(event):
    x = int(event['relativeX'])
    y = int(event['relativeY'])
    
    # Store pending click data (do not save to disk yet)
    pending_data['image'] = Car.obs.copy()
    pending_data['x'] = x
    pending_data['y'] = y
    
    # Display preview snapshot with green circle on the right widget
    snapshot = pending_data['image'].copy()
    snapshot = cv2.circle(snapshot, (x, y), 8, (0, 255, 0), 3)
    snapshot_widget.value = bgr8_to_jpeg(snapshot)
    
    # Enable Save Snapshot button
    save_button.disabled = False

im_event.on_dom_event(handle_click)

data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([camera_widget, snapshot_widget]),
    dataset_widget,
    category_widget,
    count_widget,
    controls_box
])

display(data_collection_widget)

INFO:gym_donkeycar.core.client:connecting to localhost:9091 
INFO:gym_donkeycar.envs.donkey_sim:on need car config
INFO:gym_donkeycar.envs.donkey_sim:sending car config.
INFO:gym_donkeycar.envs.donkey_sim:sim started!


starting DonkeyGym env
Setting default: start_delay 5.0
Setting default: max_cte 8.0
Setting default: frame_skip 1
Setting default: cam_resolution (120, 160, 3)
Setting default: log_level 20
Setting default: host localhost
Setting default: port 9091
Setting default: steer_limit 1.0
Setting default: throttle_min 0.0
Setting default: throttle_max 1.0


In [4]:
import torch
import torchvision

device = torch.device('cuda')
output_dim = 2 * len(dataset.categories)  # x, y coordinate for each category

# ALEXNET
# model = torchvision.models.alexnet(pretrained=True)
# model.classifier[-1] = torch.nn.Linear(4096, output_dim)

# SQUEEZENET 
# model = torchvision.models.squeezenet1_1(pretrained=True)
# model.classifier[1] = torch.nn.Conv2d(512, output_dim, kernel_size=1)
# model.num_classes = len(dataset.categories)

# RESNET 18
model = torchvision.models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, output_dim)

# RESNET 34
# model = torchvision.models.resnet34(pretrained=True)
# model.fc = torch.nn.Linear(512, output_dim)

# DENSENET 121
# model = torchvision.models.densenet121(pretrained=True)
# model.classifier = torch.nn.Linear(model.num_features, output_dim)

model = model.to(device)

model_save_button = ipywidgets.Button(description='save model')
model_load_button = ipywidgets.Button(description='load model')
model_path_widget = ipywidgets.Text(description='model path', value='road_following_model.pth')

def load_model(c):
    model.load_state_dict(torch.load(model_path_widget.value))
model_load_button.on_click(load_model)
    
def save_model(c):
    torch.save(model.state_dict(), model_path_widget.value)
model_save_button.on_click(save_model)

model_widget = ipywidgets.VBox([
    model_path_widget,
    ipywidgets.HBox([model_load_button, model_save_button])
])


display(model_widget)

d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### Live Execution

In [5]:
import threading
import time
from utils import preprocess, bgr8_to_jpeg
import torch.nn.functional as F

state_widget = ipywidgets.ToggleButtons(options=['stop', 'live'], description='state', value='stop')
prediction_widget = ipywidgets.Image(format='jpeg', width=Car.obs.shape[1], height=Car.obs.shape[0])

def live(state_widget, model, camera, prediction_widget):
    global dataset
    while state_widget.value == 'live':
        image = camera.obs
        preprocessed = preprocess(image)
        output = model(preprocessed).detach().cpu().numpy().flatten()
        category_index = dataset.categories.index(category_widget.value)
        x = output[2 * category_index]
        y = output[2 * category_index + 1]
        
        x = int(camera.obs.shape[1] * (x / 2.0 + 0.5))
        y = int(camera.obs.shape[0] * (y / 2.0 + 0.5))
        
        prediction = image.copy()
        prediction = cv2.circle(prediction, (x, y), 8, (255, 0, 0), 3)
        prediction_widget.value = bgr8_to_jpeg(prediction)
            
def start_live(change):
    if change['new'] == 'live':
        execute_thread = threading.Thread(target=live, args=(state_widget, model, Car, prediction_widget))
        execute_thread.start()

state_widget.observe(start_live, names='value')

live_execution_widget = ipywidgets.VBox([
    prediction_widget,
    state_widget
])

display(live_execution_widget)

In [6]:
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
MOMENTUM = 0.9

optimizer = torch.optim.Adam(model.parameters())
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

epochs_widget = ipywidgets.IntText(description='epochs', value=1)
eval_button = ipywidgets.Button(description='evaluate')
train_button = ipywidgets.Button(description='train')
loss_widget = ipywidgets.FloatText(description='loss')
progress_widget = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')

def train_eval(is_training):
    global BATCH_SIZE, LEARNING_RATE, MOMENTUM, model, dataset, optimizer, eval_button, train_button, loss_widget, progress_widget, state_widget
    
    try:
        train_loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        state_widget.value = 'stop'
        train_button.disabled = True
        eval_button.disabled = True
        time.sleep(1)

        if is_training:
            model = model.train()
        else:
            model = model.eval()

        while epochs_widget.value > 0:
            i = 0
            sum_loss = 0.0
            error_count = 0.0
            for images, category_idx, xy in iter(train_loader):
                # send data to device
                images = images.to(device)
                xy = xy.to(device)

                if is_training:
                    # zero gradients of parameters
                    optimizer.zero_grad()

                # execute model to get outputs
                outputs = model(images)

                # compute MSE loss over x, y coordinates for associated categories
                loss = 0.0
                for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                    loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx+2] - xy[batch_idx])**2)
                loss /= len(category_idx)

                if is_training:
                    # run backpropogation to accumulate gradients
                    loss.backward()

                    # step optimizer to adjust parameters
                    optimizer.step()

                # increment progress
                count = len(category_idx.flatten())
                i += count
                sum_loss += float(loss)
                progress_widget.value = i / len(dataset)
                loss_widget.value = sum_loss / i
                
            if is_training:
                epochs_widget.value = epochs_widget.value - 1
            else:
                break
    except Exception as e:
        print(f"Error during training/eval: {e}")
    model = model.eval()

    train_button.disabled = False
    eval_button.disabled = False
    state_widget.value = 'live'
    
train_button.on_click(lambda c: train_eval(is_training=True))
eval_button.on_click(lambda c: train_eval(is_training=False))
    
train_eval_widget = ipywidgets.VBox([
    epochs_widget,
    progress_widget,
    loss_widget,
    ipywidgets.HBox([train_button, eval_button])
])

display(train_eval_widget)

### All together!

The following widget can be used to label a multi-class x, y dataset.  It supports labeling only one instance of each class per image (ie: only one dog), but multiple classes (ie: dog, cat, horse) per image are possible.

Click the image on the top left to save an image of ``category`` to ``dataset`` at the clicked location.

| Widget | Description |
|--------|-------------|
| dataset | Selects the active dataset |
| category | Selects the active category |
| epochs | Sets the number of epochs to train for |
| train | Trains on the active dataset for the number of epochs specified |
| evaluate | Evaluates the accuracy on the active dataset over one epoch |
| model path | Sets the active model path |
| load | Loads a model from the active model path |
| save | Saves a model to the active model path |
| stop | Disables the live demo |
| live | Enables the live demo |

In [7]:
all_widget = ipywidgets.VBox([
    ipywidgets.HBox([data_collection_widget, live_execution_widget]), 
    train_eval_widget,
    model_widget
])

display(all_widget)